# 12 — Advanced Types

By this point you've used generics, variance, bounds, and givens — enough to write most of the Scala you'll meet day to day. This notebook adds the smaller, sharper tools that Scala 3 introduces on top of that foundation. None of them are required to write working code. All of them, used well, make wrong code stop compiling.

The toolbox:

- **Type aliases** — shorter names for long types.
- **Opaque types** — distinct types at compile time, free at runtime.
- **Literal & singleton types** — using a specific value as a type.
- **Union types** (`A | B`) — "either of" without wrapping.
- **Intersection types** (`A & B`) — "both of" combined.
- **Top & bottom types** — `Any`, `AnyVal`, `AnyRef`, `Nothing`, `Null`.
- **Match types** — type-level pattern matching.
- **Structural types** — typing by shape.

The goal isn't to use everything here in every program. The goal is to recognise each tool when you see it in a library, and reach for the right one when modelling a domain.

## Type aliases — naming

A type alias gives an existing type a second name. It introduces no new type — the alias and the original are *exactly the same* type to the compiler. The only purpose is readability.

In [ ]:
type UserId = Long
type Headers = Map[String, List[String]]

val id: UserId = 42L
val h: Headers = Map("content-type" -> List("application/json"))

// Aliases are transparent — UserId is Long, full stop.
val plain: Long = id    // ok, no conversion

Aliases are best used to *shorten* a long type. The trap is using them to *strengthen* a weak one — writing `type UserId = Long` does **not** stop a caller from passing any other `Long` where a `UserId` is expected. For that you want an opaque type.

## Opaque types — newtypes without overhead

A real-world problem first. Code that traffics in `Long`-typed IDs, `String`-typed emails, and `Int`-typed amounts can mix them up at the call site with no compiler complaint. `transfer(amount, userId)` and `transfer(userId, amount)` both type-check when both are `Long`. This is *primitive obsession*, and it causes a steady stream of subtle bugs.

An **opaque type** is the fix. It declares a new type that the compiler treats as distinct from its underlying representation, while still compiling down to the underlying type at runtime — zero allocation, zero indirection.

In [ ]:
object Ids:
  opaque type UserId  = Long
  opaque type OrderId = Long

  object UserId:
    def apply(n: Long): UserId = n
    extension (id: UserId) def value: Long = id

  object OrderId:
    def apply(n: Long): OrderId = n
    extension (id: OrderId) def value: Long = id

import Ids.*

val u: UserId  = UserId(1)
val o: OrderId = OrderId(2)

// u + o          // does not compile — UserId and OrderId are unrelated outside Ids
u.value + o.value  // 3 — extension methods give controlled access to the underlying Long

Three things to register about opaque types:

- **Inside** the defining scope (here `object Ids`), `UserId` and `Long` are interchangeable. The implementation can freely convert between them.
- **Outside** that scope, they are completely distinct. Passing a raw `Long` where a `UserId` is expected fails to compile.
- At **runtime** there's no wrapper class. A `UserId` is, in memory, just a `Long`. You get type safety with no boxing cost — which matters in hot loops and large collections.

This is the same idea as Haskell's `newtype` or Rust's tuple-struct wrapper, with the Scala-specific twist that the scope of "opaque to whom" is the enclosing scope of the definition.

## Literal types — a specific value as a type

Most types describe *kinds* of values — `Int` describes all integers. A **literal type** describes a single one. The literal `42` is also a type, written `42`, inhabited by exactly one value: `42`.

In [ ]:
val answer: 42 = 42
// val wrong: 42 = 43     // does not compile — 43 is not of type 42

type YesNo = "yes" | "no"
val a: YesNo = "yes"
val b: YesNo = "no"
// val c: YesNo = "maybe" // does not compile

Literal types shine in two places. First, as the building blocks of *string-enum-like* APIs — the `"yes" | "no"` above is a tiny union of two literal types. Second, in libraries that compute types from values — a request handler that promises to return JSON might be typed `Handler["application/json"]`. You rarely declare them by hand outside of API boundaries.

## Singleton types — `.type`

Closely related: every value has a singleton type accessible via `.type`. It describes the one-element set containing just that value.

In [ ]:
object Logger

val l: Logger.type = Logger    // the singleton type of the Logger object
// val other: Logger.type = new Object   // does not compile — only Logger itself fits

You'll meet singleton types most often as the type of `object` declarations — `Logger.type` is the type of the `Logger` object itself. They also show up in DSLs and path-dependent typing, but those are niche.

## Union types — `A | B`

A **union type** `A | B` is the type of values that are either an `A` or a `B`. Unlike `Either`, there's no wrapping — no `Left` or `Right` constructor. You just have a value whose static type is the union.

This is brand-new in Scala 3 and the most useful headline feature of the type system.

In [ ]:
def describe(x: Int | String): String = x match
  case n: Int    => s"int $n"
  case s: String => s"string of length ${s.length}"

describe(42)        // "int 42"
describe("scala")   // "string of length 5"

When you `match` on a union, the compiler narrows each branch to the specific member type. So in the `case n: Int` branch, `n` is an `Int`, full stop — no cast, no run-time check needed beyond the match itself.

Common uses for unions:

- **Return-or-error without `Either`.** A parser can return `Token | ParseError` and let the caller handle each.
- **Heterogeneous parameters.** A logging function that accepts `String | Throwable`.
- **Literal enums.** `"GET" | "POST" | "PUT" | "DELETE"` is a perfectly good HTTP method type.

**Union vs Either.** Reach for `Either` when *which side it is* carries meaning (success vs failure) and you want to chain with `map`/`flatMap`. Reach for a union when the alternatives are simply different shapes the same value can take.

## Intersection types — `A & B`

The mirror image. `A & B` is the type of values that satisfy *both* `A` and `B` — they implement everything in `A` *and* everything in `B`.

In [ ]:
trait Logger:
  def log(msg: String): Unit

trait Metrics:
  def count(name: String): Unit

def instrumented(svc: Logger & Metrics): Unit =
  svc.log("starting")
  svc.count("requests")

A parameter typed `Logger & Metrics` requires the caller to supply something that is *both* a logger and a metrics collector. Inside the function, all methods from both traits are available without any casting. This is the modern replacement for `with`-style trait composition in signatures — same idea, cleaner notation.

### Algebraic intuition

Unions and intersections form an algebra on types that mirrors `or`/`and` on booleans. `A | A` is `A`. `A & A` is `A`. `A | B` is the same as `B | A` (the order doesn't matter). Likewise for `&`. Intersections distribute over unions. This algebra is the basis for *match types* coming up below — types that compute based on the shape of other types.

## Top and bottom types

Scala's type hierarchy has a shape worth picturing in one diagram:

```
                       Any
                      /   \
                 AnyVal   AnyRef
                 |   |    |   |
                Int Bool String User ...
                      \   /
                       Null         (subtype of every AnyRef)
                        |
                      Nothing       (subtype of every type)
```

- `Any` — the **top type**. Every value is an `Any`.
- `AnyVal` — JVM primitives (`Int`, `Double`, `Boolean`, `Char`, `Unit`...).
- `AnyRef` — reference types (anything from `java.lang.Object` downwards). Equivalent to Java's `Object`.
- `Null` — the type of the `null` literal. Subtype of every reference type but not `AnyVal`. With Scala 3's null-safe mode (`-Yexplicit-nulls`), this connection is severed.
- `Nothing` — the **bottom type**. A subtype of every type. It has *no values*.

`Nothing` looks useless — a type you can't construct. Its job is to represent *no value at all*, which makes it the return type of functions that never return (because they always throw or loop forever) and the element type of empty generic containers like `Nil` and `None`.

In [ ]:
def crash(msg: String): Nothing = throw new RuntimeException(msg)

// Because Nothing <: every type, crash can stand in anywhere.
val n: Int    = if (true) 1 else crash("unreachable")
val s: String = if (true) "ok" else crash("unreachable")

// And `Nil: List[Nothing]` is assignable to any `List[A]`:
val xs: List[Int] = Nil    // works because Nothing is a subtype of Int (for List purposes via covariance)

That last assignment combines `Nothing` with the *covariance* of `List` you saw in notebook 10. `Nil` has type `List[Nothing]`. Because `List` is covariant and `Nothing <: Int`, `List[Nothing] <: List[Int]`. The empty list works as the empty version of every list type at once — and that's only possible because `Nothing` sits at the bottom of the hierarchy.

## Match types — pattern matching at the type level

Match types let you compute a *type* by pattern-matching on another type. They're niche, and you mostly read them rather than write them, but understanding the shape is helpful when a library leans on them.

In [ ]:
type Element[X] = X match
  case String      => Char
  case Array[t]    => t
  case Iterable[t] => t

summon[Element[String]      =:= Char]    // ok
summon[Element[Array[Int]]  =:= Int]     // ok
summon[Element[List[Boolean]] =:= Boolean] // ok

Read it like a value-level `match`, but on the right side of an `=` and operating on types. `Element[String]` *reduces* to `Char`. `Element[List[Boolean]]` reduces to `Boolean`. The `=:=` operator is the standard library's type-equality witness — `summon[A =:= B]` compiles only if `A` and `B` are the same type after all reductions.

Match types make APIs like `tupleHead`, `init`/`last` on tuples, and various "map a tuple of types" combinators possible. You'll meet them in advanced library code, especially around generic derivation. In application code, leave them alone unless you have a clear reason.

## Structural types — typing by shape

A structural type describes what methods an object must have, without naming a specific class or trait. It's nominal Scala's escape hatch into duck typing — useful when interacting with reflection-heavy APIs but rare elsewhere.

```
  def closeQuietly(r: { def close(): Unit }): Unit =
    try r.close()
    catch case _: Throwable => ()
```

Any value with a `close(): Unit` method matches, regardless of whether it implements a `Closeable` trait. Behind the scenes the JVM uses reflection to make the call, which is why this carries a performance cost and isn't the default. Mention it here so you recognise it; reach for nominal types (traits) in almost every real situation.

## Putting it together — a domain-modelled API

Most of these features earn their keep when combined. A small example pulls together opaque types, a union, an extension, and an intersection.

In [ ]:
object Money:
  opaque type Cents = Long
  object Cents:
    def apply(n: Long): Cents = n
    extension (c: Cents)
      def +(other: Cents): Cents = c + other
      def toDollars: Double      = c / 100.0

import Money.*

enum AuthError:
  case Unauthenticated
  case Forbidden(reason: String)

case class Account(id: Long, balance: Cents)

// A union return type — either the account, or one of two named errors.
def lookup(token: String): Account | AuthError =
  if token.isEmpty then AuthError.Unauthenticated
  else if token == "bad" then AuthError.Forbidden("revoked")
  else Account(1, Cents(5000))

lookup("good") match
  case acct: Account              => s"balance: ${acct.balance.toDollars}"
  case AuthError.Unauthenticated  => "please log in"
  case AuthError.Forbidden(why)   => s"forbidden: $why"

Notice the three features doing different jobs:

- **Opaque `Cents`** stops anyone from accidentally storing dollars or millicents in the same field. Inside `Money`, it's a plain `Long`. Outside, it's a distinct type with controlled operations.
- **Union `Account | AuthError`** lets the function return either a success or a structured error *without* wrapping in `Either`. The caller pattern-matches and is checked for exhaustiveness because `AuthError` is sealed.
- **Extension methods on `Cents`** provide `+` and `toDollars` without making `Cents` a wrapper class. At runtime, the addition is a primitive `Long` add.

Used together, this is what modern Scala 3 domain modelling looks like — precise types at compile time, plain primitives at runtime.

## What's next

Notebook 13 leaves the type system behind and moves to runtime concerns: **Futures & concurrency**. `Future[A]` is a `map`/`flatMap`-shaped container just like the ones from notebook 09, but its values are produced *eventually* — so composition rules carry over while the semantics shift. After that, notebook 14 brings the error-handling and resource-management story together, and notebook 15 lands at the Spark-shaped use of everything you've built up.